# Did a gameplay change improve player performance?

A game team enabled a change for Group A. Group B did not receive it. This notebook demonstrates a before-and-after comparison that accounts for shared time changes and repeated battles per player. All records are synthetic. They do not reproduce the assessment game's results.

## 1. Generate synthetic battle records

Group A starts with lower average performance. Both groups experience a shared time change. The simulated feature adds 25 damage points per battle for Group A after rollout. This known value is used only to demonstrate the method.

In [ ]:
import random
from collections import defaultdict

random.seed(420)
N_PER_GROUP = 400
BATTLES_PER_PERIOD = 8
rows = []

for group in ('A', 'B'):
    for player_number in range(N_PER_GROUP):
        player_id = f'SYN-{group}-{player_number:04d}'
        player_effect = random.gauss(0, 80)
        group_baseline = -35 if group == 'A' else 0
        for period in ('pre', 'post'):
            for battle_number in range(BATTLES_PER_PERIOD):
                post = period == 'post'
                feature_effect = 25 if group == 'A' and post else 0
                rows.append({
                    'player_id': player_id,
                    'group': group,
                    'period': period,
                    'damage': 1000 + group_baseline + player_effect - 20 * post + feature_effect + random.gauss(0, 180),
                    'kills': 0.72 - 0.05 * (group == 'A') - 0.02 * post + 0.01 * (group == 'A' and post) + random.gauss(0, 0.45),
                    'battle_seconds': 350 + 5 * (group == 'A') - 5 * post + random.gauss(0, 70)
                })

print(f'Synthetic battle rows: {len(rows):,}')
print(f'Synthetic players: {2 * N_PER_GROUP:,}')

## 2. Compare group averages

A single comparison between Group A and Group B mixes baseline group differences with any change after rollout. Show all four group-by-period averages before drawing a conclusion.

In [ ]:
group_period = defaultdict(lambda: defaultdict(list))
for row in rows:
    for metric in ('damage', 'kills', 'battle_seconds'):
        group_period[(row['group'], row['period'])][metric].append(row[metric])

for metric in ('damage', 'kills', 'battle_seconds'):
    print(f'\n{metric}')
    for group in ('A', 'B'):
        means = {period: sum(group_period[(group, period)][metric]) / len(group_period[(group, period)][metric]) for period in ('pre', 'post')}
        print(f"Group {group}: pre={means['pre']:.2f}, post={means['post']:.2f}, change={means['post'] - means['pre']:.2f}")

## 3. Estimate the difference in differences

First compute each player's change from pre to post. Then compare the average player-level change in Group A with the average change in Group B. Resampling players keeps each player's repeated battles together.

In [ ]:
player_values = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list))))
for row in rows:
    for metric in ('damage', 'kills', 'battle_seconds'):
        player_values[row['group']][row['player_id']][metric][row['period']].append(row[metric])

def player_change(group, metric):
    changes = []
    for metrics_by_period in player_values[group].values():
        pre = sum(metrics_by_period[metric]['pre']) / len(metrics_by_period[metric]['pre'])
        post = sum(metrics_by_period[metric]['post']) / len(metrics_by_period[metric]['post'])
        changes.append(post - pre)
    return changes

def mean(values):
    return sum(values) / len(values)

def did(metric):
    change_a = player_change('A', metric)
    change_b = player_change('B', metric)
    estimate = mean(change_a) - mean(change_b)
    bootstrap = []
    for _ in range(1000):
        sample_a = [random.choice(change_a) for _ in change_a]
        sample_b = [random.choice(change_b) for _ in change_b]
        bootstrap.append(mean(sample_a) - mean(sample_b))
    bootstrap.sort()
    low = bootstrap[int(0.025 * len(bootstrap))]
    high = bootstrap[int(0.975 * len(bootstrap))]
    return estimate, low, high

for metric in ('damage', 'kills', 'battle_seconds'):
    estimate, low, high = did(metric)
    print(f'{metric}: difference-in-differences={estimate:.3f}, bootstrap 95% interval=({low:.3f}, {high:.3f})')

## 4. Interpret the estimate carefully

The synthetic data were generated with a 25-point damage effect. The difference-in-differences estimate should be near that value. This confirms that the code follows the simulation design. It does not validate the method on the original game data.

A real rollout comparison needs these checks:

- Confirm that group membership stayed fixed and that the rollout date is correct.
- Compare several pre-rollout periods to assess whether group trends were similar.
- Check for other releases or events that affected Group A differently.
- Define one primary outcome and guardrail metrics before analysis.
- Keep all battles from one player together when estimating uncertainty.
- If assignment or trend assumptions are weak, report an association and run a randomized test.

## Takeaway

A small p-value from a pooled battle-level comparison does not isolate a feature effect. Compare the groups' changes over time and account for repeated battles per player.